# 🎓 Classroom Face Recognition - ArcFace Embedding Generator
## Chạy trên Google Colab GPU để tạo face embeddings chất lượng cao

**Pipeline:**
1. Upload folder ảnh học sinh (zip)
2. InsightFace buffalo_l → Detect + Align + Embed mỗi khuôn mặt
3. Xuất `deep_embeddings.pkl` → Download về máy
4. Copy vào `data/face_embeddings/deep_embeddings.pkl`

**Cấu trúc thư mục ảnh:**
```
photos/
├── HS001_NguyenVanAn/
│   ├── img1.jpg
│   ├── img2.jpg
│   └── img3.jpg
├── HS002_PhamThiBich/
│   └── img1.jpg
```
Tên folder = `{student_id}_{ten_hoc_sinh}` (hoặc chỉ `{student_id}`)

In [ ]:
# ===== BƯỚC 1: KIỂM TRA GPU =====
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU available:')
    print(result.stdout[:500])
else:
    print('⚠️  Không có GPU! Colab → Runtime → Change runtime type → GPU T4')
    print('CPU mode vẫn chạy được nhưng chậm hơn 10x')

In [ ]:
# ===== BƯỚC 2: CÀI ĐẶT THƯ VIỆN =====
print('📦 Installing dependencies...')
!pip install -q insightface==0.7.3 onnxruntime-gpu onnx opencv-python-headless
!pip install -q Pillow scikit-learn tqdm matplotlib
print('✅ Done!')

In [ ]:
# ===== BƯỚC 3: IMPORT =====
import os
import cv2
import pickle
import numpy as np
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import insightface
from insightface.app import FaceAnalysis
from google.colab import files
import datetime

print('✅ Imports OK')
print(f'InsightFace version: {insightface.__version__}')

In [ ]:
# ===== BƯỚC 4: LOAD ARCFACE MODEL =====
# buffalo_l: Best accuracy model (ResNet100 backbone)
# buffalo_s: Lighter model nếu RAM ít
print('🔄 Loading InsightFace buffalo_l model...')
print('(Lần đầu download ~200MB, các lần sau dùng cache)')

app = FaceAnalysis(
    name='buffalo_l',    # best accuracy model
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
app.prepare(ctx_id=0, det_size=(640, 640))

print('✅ ArcFace model loaded!')
print(f'Detection model: {app.det_model.taskname}')

In [ ]:
# ===== BƯỚC 5: UPLOAD ẢNH HỌC SINH =====
# Cách 1: Upload file ZIP
print('📁 Upload file ZIP chứa thư mục ảnh học sinh...')
print('Cấu trúc: photos/HS001_TenHS/[anh1.jpg, anh2.jpg, ...]')

uploaded = files.upload()

# Giải nén
WORK_DIR = Path('/content/classroom_faces')
WORK_DIR.mkdir(exist_ok=True)

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f'📦 Extracting {filename}...')
        with zipfile.ZipFile(filename, 'r') as zf:
            zf.extractall(WORK_DIR)
        print(f'✅ Extracted to {WORK_DIR}')

# Liệt kê thư mục học sinh
student_dirs = [d for d in WORK_DIR.rglob('*') if d.is_dir() and not d.name.startswith('.')]
# Tìm thư mục có ảnh
student_dirs = [d for d in student_dirs if any(f.suffix.lower() in ['.jpg', '.jpeg', '.png'] for f in d.iterdir() if f.is_file())]

print(f'\n📊 Tìm thấy {len(student_dirs)} thư mục học sinh:')
for d in student_dirs[:10]:
    imgs = [f for f in d.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']]
    print(f'  {d.name}: {len(imgs)} ảnh')
if len(student_dirs) > 10:
    print(f'  ... và {len(student_dirs)-10} thư mục khác')

In [ ]:
# ===== (TÙY CHỌN) Tạo ảnh mẫu để test =====
# Bỏ qua nếu bạn đã upload ảnh thật

CREATE_DEMO = False  # ← Đổi thành True nếu muốn demo

if CREATE_DEMO:
    import urllib.request
    # Tải ảnh demo từ thư viện public face dataset
    demo_students = {
        'HS001_Demo': 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/220px-Gatto_europeo4.jpg'
    }
    for folder, url in demo_students.items():
        (WORK_DIR / folder).mkdir(exist_ok=True)
        urllib.request.urlretrieve(url, WORK_DIR / folder / 'img1.jpg')
    print('✅ Demo images created')

In [ ]:
# ===== BƯỚC 6: HÀM TIỆN ÍCH =====

def parse_student_id_name(folder_name: str):
    """
    Parse tên thư mục → (student_id, name)
    Ví dụ:
      'HS001_NguyenVanAn' → ('HS001', 'Nguyen Van An')
      'HS002'             → ('HS002', 'HS002')
      'HS003_PhamThiBich' → ('HS003', 'Pham Thi Bich')
    """
    parts = folder_name.split('_', 1)
    student_id = parts[0]
    if len(parts) > 1:
        # Convert CamelCase or underscores to readable name
        name_raw = parts[1].replace('_', ' ')
        name = ' '.join(w.capitalize() for w in name_raw.split())
    else:
        name = student_id
    return student_id, name


def load_image(path: Path) -> np.ndarray:
    """Load image, convert to RGB numpy array."""
    img = Image.open(path).convert('RGB')
    # Resize if too large (>2MP)
    if img.width * img.height > 2_000_000:
        scale = (2_000_000 / (img.width * img.height)) ** 0.5
        img = img.resize((int(img.width * scale), int(img.height * scale)), Image.LANCZOS)
    return np.array(img)


def get_face_embedding(img_rgb: np.ndarray, student_id: str, img_name: str):
    """
    Detect face + generate ArcFace 512-dim embedding.
    Returns: (embedding, face_bbox, quality_score) or None nếu không detect được
    """
    faces = app.get(img_rgb)
    
    if len(faces) == 0:
        return None, f'{student_id}/{img_name}: Không detect được khuôn mặt'
    
    if len(faces) > 1:
        # Lấy khuôn mặt lớn nhất (gần camera nhất)
        faces = sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]), reverse=True)
        print(f'  ⚠️  {student_id}/{img_name}: {len(faces)} mặt, lấy mặt lớn nhất')
    
    face = faces[0]
    embedding = face.normed_embedding  # already L2-normalized, shape (512,)
    det_score = float(face.det_score)
    
    if det_score < 0.5:
        return None, f'{student_id}/{img_name}: Det score thấp ({det_score:.2f}) - ảnh chất lượng thấp'
    
    return {
        'embedding': embedding,
        'bbox': face.bbox.tolist(),
        'det_score': det_score,
        'img_name': img_name,
    }, None


print('✅ Helper functions defined')

In [ ]:
# ===== BƯỚC 7: BATCH EMBEDDING GENERATION =====
print('🚀 Bắt đầu tạo Face Embeddings...')
print('='*60)

SUPPORTED_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

deep_embeddings = {}   # Kết quả cuối cùng
failed_images = []     # Ảnh không processing được
stats = {'total_imgs': 0, 'success': 0, 'failed': 0, 'students': 0}

for student_dir in tqdm(student_dirs, desc='Học sinh'):
    student_id, name = parse_student_id_name(student_dir.name)
    
    # Lấy tất cả ảnh
    img_files = sorted([
        f for f in student_dir.iterdir()
        if f.suffix.lower() in SUPPORTED_EXTS and f.is_file()
    ])
    
    if not img_files:
        print(f'⚠️  {student_id}: Không có ảnh nào')
        continue
    
    student_results = {
        'name': name,
        'student_id': student_id,
        'embeddings': [],        # List of 512-dim numpy arrays
        'embedding_meta': [],    # Thông tin từng embedding
        'enrolled_at': datetime.datetime.now().isoformat(),
        'source': 'colab_arcface_buffalo_l',
        'model_version': 'buffalo_l_v1',
    }
    
    for img_file in img_files:
        stats['total_imgs'] += 1
        try:
            img_rgb = load_image(img_file)
            result, error = get_face_embedding(img_rgb, student_id, img_file.name)
            
            if result:
                student_results['embeddings'].append(result['embedding'])
                student_results['embedding_meta'].append({
                    'img_name': result['img_name'],
                    'det_score': result['det_score'],
                    'bbox': result['bbox'],
                })
                stats['success'] += 1
            else:
                failed_images.append(error)
                stats['failed'] += 1
        except Exception as e:
            error_msg = f'{student_id}/{img_file.name}: Exception: {str(e)}'
            failed_images.append(error_msg)
            stats['failed'] += 1
    
    if student_results['embeddings']:
        # Tính centroid embedding (mean của tất cả embeddings)
        centroid = np.mean(student_results['embeddings'], axis=0)
        centroid = centroid / np.linalg.norm(centroid)  # Re-normalize
        student_results['centroid'] = centroid
        
        deep_embeddings[student_id] = student_results
        stats['students'] += 1

# Kết quả
print('\n' + '='*60)
print('📊 KẾT QUẢ:')
print(f'  ✅ Học sinh đã xử lý: {stats["students"]}')
print(f'  ✅ Ảnh thành công: {stats["success"]}/{stats["total_imgs"]}')
print(f'  ❌ Ảnh lỗi: {stats["failed"]}')

if failed_images:
    print('\n⚠️  Ảnh không xử lý được:')
    for msg in failed_images[:10]:
        print(f'  - {msg}')

In [ ]:
# ===== BƯỚC 8: VISUALIZE KẾT QUẢ =====
print('📊 Visualizing embedding quality...')

if len(deep_embeddings) >= 2:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    
    all_embeddings = []
    all_labels = []
    all_names = []
    
    for sid, data in deep_embeddings.items():
        for emb in data['embeddings']:
            all_embeddings.append(emb)
            all_labels.append(sid)
            all_names.append(data['name'])
    
    all_embeddings = np.array(all_embeddings)
    
    if len(all_embeddings) >= 3:
        # PCA 2D
        pca = PCA(n_components=2)
        emb_2d = pca.fit_transform(all_embeddings)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        unique_ids = list(set(all_labels))
        colors = plt.cm.tab20(np.linspace(0, 1, len(unique_ids)))
        
        for i, sid in enumerate(unique_ids[:20]):  # Max 20 students
            mask = [l == sid for l in all_labels]
            pts = emb_2d[mask]
            ax.scatter(pts[:, 0], pts[:, 1], c=[colors[i]], label=sid, s=100, alpha=0.8)
        
        ax.set_title('Face Embeddings PCA 2D\n(Cùng màu = cùng học sinh — tốt nếu cluster rõ ràng)', fontsize=14)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        plt.tight_layout()
        plt.savefig('/content/embeddings_pca.png', dpi=100, bbox_inches='tight')
        plt.show()
        print('✅ PCA visualization saved')
    else:
        print('⚠️  Cần ít nhất 3 embeddings để visualize')
else:
    print('⚠️  Cần ít nhất 2 học sinh để visualize')

In [ ]:
# ===== BƯỚC 9: ĐÁNH GIÁ ACCURACY (nếu có >1 ảnh/học sinh) =====
print('🎯 Self-evaluation: Tự kiểm tra khả năng phân biệt...')

from sklearn.metrics.pairwise import cosine_similarity

THRESHOLD = 0.45  # Ngưỡng nhận diện (tùy chỉnh)

true_positives = 0
false_positives = 0
true_negatives = 0
false_negatives = 0
total_pairs = 0

student_ids = list(deep_embeddings.keys())

for i, sid1 in enumerate(student_ids):
    data1 = deep_embeddings[sid1]
    if len(data1['embeddings']) < 2:
        continue
    
    # Same-person pairs (should be HIGH similarity)
    for j in range(len(data1['embeddings'])):
        for k in range(j+1, len(data1['embeddings'])):
            sim = float(cosine_similarity(
                data1['embeddings'][j].reshape(1, -1),
                data1['embeddings'][k].reshape(1, -1)
            ))
            total_pairs += 1
            if sim >= THRESHOLD:
                true_positives += 1
            else:
                false_negatives += 1

    # Different-person pairs (should be LOW similarity)
    for sid2 in student_ids[i+1:i+4]:  # Sample vài học sinh khác
        data2 = deep_embeddings[sid2]
        if data1['embeddings'] and data2['embeddings']:
            sim = float(cosine_similarity(
                data1['embeddings'][0].reshape(1, -1),
                data2['embeddings'][0].reshape(1, -1)
            ))
            total_pairs += 1
            if sim < THRESHOLD:
                true_negatives += 1
            else:
                false_positives += 1

if total_pairs > 0:
    precision = true_positives / max(true_positives + false_positives, 1)
    recall = true_positives / max(true_positives + false_negatives, 1)
    accuracy = (true_positives + true_negatives) / total_pairs
    
    print(f'\n📊 KẾT QUẢ SELF-EVALUATION (threshold={THRESHOLD}):')
    print(f'  Accuracy:  {accuracy:.1%}')
    print(f'  Precision: {precision:.1%}')
    print(f'  Recall:    {recall:.1%}')
    print(f'  Cặp kiểm tra: {total_pairs}')
    
    if accuracy >= 0.90:
        print('  👍 Embeddings chất lượng rất tốt!')
    elif accuracy >= 0.75:
        print('  ⚠️  Chấp nhận được. Thêm ảnh/học sinh để cải thiện.')
    else:
        print('  ❌ Chất lượng thấp. Kiểm tra lại ảnh (ánh sáng, góc chụp)')
else:
    print('⚠️  Cần ít nhất 2 ảnh/học sinh để đánh giá')

In [ ]:
# ===== BƯỚC 10: LƯU VÀ DOWNLOAD =====
OUTPUT_FILE = '/content/deep_embeddings.pkl'

# Chuẩn bị metadata
output_data = {
    'version': '2.0',
    'model': 'insightface_buffalo_l_arcface',
    'embedding_dim': 512,
    'threshold_recommended': 0.45,
    'created_at': datetime.datetime.now().isoformat(),
    'total_students': len(deep_embeddings),
    'total_embeddings': sum(len(v['embeddings']) for v in deep_embeddings.values()),
    'students': deep_embeddings,
}

with open(OUTPUT_FILE, 'wb') as f:
    pickle.dump(output_data, f, protocol=pickle.HIGHEST_PROTOCOL)

# Kiểm tra file size
file_size = os.path.getsize(OUTPUT_FILE) / 1024
print(f'\n✅ Đã lưu: {OUTPUT_FILE}')
print(f'   Kích thước: {file_size:.1f} KB')
print(f'   Học sinh: {len(deep_embeddings)}')
print(f'   Tổng embeddings: {output_data["total_embeddings"]}')

print('\n📥 Đang tải file về máy...')
files.download(OUTPUT_FILE)
print('\n🎉 XONG! Tiếp theo:')
print('   1. Copy file deep_embeddings.pkl vào thư mục:')
print('      data/face_embeddings/deep_embeddings.pkl')
print('   2. Restart server (python main.py)')
print('   3. Hệ thống tự động load và dùng ArcFace embeddings!')

---
## 📝 HƯỚNG DẪN CẬP NHẬT EMBEDDINGS

### Thêm học sinh mới (không cần chạy lại toàn bộ):
```python
# Load embeddings cũ
with open('deep_embeddings.pkl', 'rb') as f:
    data = pickle.load(f)

# Thêm học sinh mới
new_photos_dir = Path('photos/HS030_HoangVanMinh')
student_id, name = parse_student_id_name(new_photos_dir.name)

embeddings = []
for img_file in new_photos_dir.iterdir():
    img_rgb = load_image(img_file)
    result, _ = get_face_embedding(img_rgb, student_id, img_file.name)
    if result:
        embeddings.append(result['embedding'])

data['students'][student_id] = {
    'name': name,
    'embeddings': embeddings,
    'centroid': np.mean(embeddings, axis=0),
}

with open('deep_embeddings.pkl', 'wb') as f:
    pickle.dump(data, f)
```

### Gợi ý số ảnh mỗi học sinh:
| Số ảnh | Accuracy | Ghi chú |
|--------|----------|----------|
| 1 ảnh | 80-85% | Tối thiểu |
| 3 ảnh | 90-94% | Khuyến nghị |
| 5+ ảnh | 95%+ | Tốt nhất |

**Gợi ý góc chụp:** Nhìn thẳng + nhìn nghiêng trái/phải nhẹ + đội mũ/đeo kính nếu hay dùng